<a href="https://colab.research.google.com/github/saravananarasan-bits-aiml/drybean-ml-classifier-sarasan/blob/main/2025AC05064_ML_Assignment2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [10]:
!pip install ucimlrepo
from ucimlrepo import fetch_ucirepo
import pandas as pd

dry_bean = fetch_ucirepo(id=602)
X = dry_bean.data.features        # 16 numeric columns
y = dry_bean.data.targets         # the 'Class' column (7 bean types)

df = pd.concat([X, y], axis=1)
print(df.shape)                    # expect (13611, 17)
print(df['Class'].value_counts())  # 7 classes + their counts
print("missing values:", df.isnull().sum().sum())   # expect 0

(13611, 17)
Class
DERMASON    3546
SIRA        2636
SEKER       2027
HOROZ       1928
CALI        1630
BARBUNYA    1322
BOMBAY       522
Name: count, dtype: int64
missing values: 0


In [11]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Flatten the target for sklearn
y = y.values.ravel()

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Apply StandardScaler
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

print("X_train:", X_train.shape, "| X_test:", X_test.shape)


X_train: (10888, 16) | X_test: (2723, 16)


In [13]:
from sklearn.linear_model import LogisticRegression

# Create the model
log_reg = LogisticRegression(max_iter=1000, random_state=42)

# Train it on the scaled training data
log_reg.fit(X_train, y_train)

# Predict on the unseen test data
y_pred_lr = log_reg.predict(X_test)

# Quick peek at how it did
from sklearn.metrics import accuracy_score
print("Logistic Regression accuracy:", round(accuracy_score(y_test, y_pred_lr), 4))

Logistic Regression accuracy: 0.9207


In [14]:
from sklearn.tree import DecisionTreeClassifier

# Create the model
dt = DecisionTreeClassifier(max_depth=8, random_state=42)

# Train
dt.fit(X_train, y_train)

# Predict on unseen test data
y_pred_dt = dt.predict(X_test)

# Quick peek
print("Decision Tree accuracy:", round(accuracy_score(y_test, y_pred_dt), 4))

Decision Tree accuracy: 0.9041


In [15]:
from sklearn.neighbors import KNeighborsClassifier

# Create the model — k = 5 neighbours
knn = KNeighborsClassifier(n_neighbors=5)

# Train (kNN just memorizes the training data — the "lazy learner")
knn.fit(X_train, y_train)

# Predict on unseen test data
y_pred_knn = knn.predict(X_test)

# Quick peek
print("kNN accuracy:", round(accuracy_score(y_test, y_pred_knn), 4))

kNN accuracy: 0.9166


In [16]:
from sklearn.naive_bayes import GaussianNB

# Create the model (no settings needed)
nb = GaussianNB()

# Train — it learns each feature's mean & spread, per class
nb.fit(X_train, y_train)

# Predict on unseen test data
y_pred_nb = nb.predict(X_test)

# Quick peek
print("Naive Bayes accuracy:", round(accuracy_score(y_test, y_pred_nb), 4))

Naive Bayes accuracy: 0.8979


In [17]:
from sklearn.ensemble import RandomForestClassifier

# Create the model — a forest of 100 trees
rf = RandomForestClassifier(n_estimators=100, random_state=42)

# Train (grows 100 diverse trees on random subsets)
rf.fit(X_train, y_train)

# Predict on unseen test data
y_pred_rf = rf.predict(X_test)

# Quick peek
print("Random Forest accuracy:", round(accuracy_score(y_test, y_pred_rf), 4))

Random Forest accuracy: 0.9207


In [18]:
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, matthews_corrcoef, roc_auc_score)
import pandas as pd

# Put all 5 trained models in a dictionary so we can loop over them
models = {
    "Logistic Regression": log_reg,
    "Decision Tree":        dt,
    "kNN":                  knn,
    "Naive Bayes":          nb,
    "Random Forest":        rf,
}

results = []

for name, model in models.items():
    # hard predictions (the class labels)
    y_pred = model.predict(X_test)
    # probability predictions (needed for AUC) — one probability per class
    y_proba = model.predict_proba(X_test)

    results.append({
        "Model":     name,
        "Accuracy":  accuracy_score(y_test, y_pred),
        "AUC":       roc_auc_score(y_test, y_proba, multi_class="ovr", average="macro"),
        "Precision": precision_score(y_test, y_pred, average="macro"),
        "Recall":    recall_score(y_test, y_pred, average="macro"),
        "F1":        f1_score(y_test, y_pred, average="macro"),
        "MCC":       matthews_corrcoef(y_test, y_pred),
    })

# Build a tidy comparison table, rounded, sorted by MCC
comparison = pd.DataFrame(results).round(4).sort_values("MCC", ascending=False)
print(comparison.to_string(index=False))

              Model  Accuracy    AUC  Precision  Recall     F1    MCC
Logistic Regression    0.9207 0.9948     0.9349  0.9314 0.9329 0.9042
      Random Forest    0.9207 0.9926     0.9355  0.9313 0.9333 0.9041
                kNN    0.9166 0.9833     0.9320  0.9271 0.9293 0.8992
      Decision Tree    0.9041 0.9822     0.9175  0.9147 0.9159 0.8841
        Naive Bayes    0.8979 0.9916     0.9112  0.9092 0.9091 0.8773


In [19]:
# Recover the RAW (unscaled) test features, aligned with the labels
X_train_raw, X_test_raw, y_tr_raw, y_te_raw = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

# Build test_data.csv: 16 raw features + the true Class label
test_df = X_test_raw.copy()
test_df["Class"] = y_te_raw
test_df.to_csv("test_data.csv", index=False)
print("test_data.csv:", test_df.shape)   # expect (2723, 17)

test_data.csv: (2723, 17)


In [20]:
import joblib, os
os.makedirs("model", exist_ok=True)

joblib.dump(scaler,  "model/scaler.pkl")
joblib.dump(log_reg, "model/logistic_regression.pkl")
joblib.dump(dt,      "model/decision_tree.pkl")
joblib.dump(knn,     "model/knn.pkl")
joblib.dump(nb,      "model/naive_bayes.pkl")
joblib.dump(rf,      "model/random_forest.pkl")

print(os.listdir("model"))   # 6 .pkl files

['scaler.pkl', 'knn.pkl', 'random_forest.pkl', 'naive_bayes.pkl', 'logistic_regression.pkl', 'decision_tree.pkl']


In [21]:
!pip install streamlit -q
!npm install -g localtunnel

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 33.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 33.8 MB/s eta 0:00:00
⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏
added 22 packages in 3s
⠏
⠏3 packages are looking for funding
⠏  run `npm fund` for details
⠏

In [35]:
%%writefile app.py
import streamlit as st
import pandas as pd
import joblib

st.title("Dry Bean Classifier")
st.write("Upload test data, pick a model, and see how it performs.")

@st.cache_resource
def load_artifacts():
    scaler = joblib.load("model/scaler.pkl")
    models = {
        "Logistic Regression": joblib.load("model/logistic_regression.pkl"),
        "Decision Tree":        joblib.load("model/decision_tree.pkl"),
        "kNN":                  joblib.load("model/knn.pkl"),
        "Naive Bayes":          joblib.load("model/naive_bayes.pkl"),
        "Random Forest":        joblib.load("model/random_forest.pkl"),
    }
    return scaler, models

scaler, models = load_artifacts()
st.success(f"Loaded {len(models)} models and the scaler.")

st.header("1. Upload test data")
uploaded = st.file_uploader("Upload your test_data.csv", type="csv")

if uploaded is not None:
    data = pd.read_csv(uploaded)
    st.write(f"Uploaded data shape: {data.shape[0]} rows, {data.shape[1]} columns")
    st.dataframe(data.head())

    st.header("2. Choose a model")
    model_name = st.selectbox("Select a model", list(models.keys()))

    X_new = data.drop(columns=["Class"])
    y_true = data["Class"]

    X_scaled = scaler.transform(X_new)

    chosen_model = models[model_name]
    y_pred = chosen_model.predict(X_scaled)

    st.success(f"Ran {model_name} on {len(y_true)} rows.")

    # ---- 3. Evaluation metrics ----
    from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                                 f1_score, matthews_corrcoef)

    st.header("3. Evaluation metrics")
    metrics = {
        "Accuracy":  accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, average="macro"),
        "Recall":    recall_score(y_true, y_pred, average="macro"),
        "F1":        f1_score(y_true, y_pred, average="macro"),
        "MCC":       matthews_corrcoef(y_true, y_pred),
    }
    metrics_df = pd.DataFrame(metrics, index=[model_name]).round(4)
    st.dataframe(metrics_df)

    # ---- 4. Confusion matrix ----
    from sklearn.metrics import confusion_matrix
    import matplotlib.pyplot as plt
    import seaborn as sns

    st.header("4. Confusion matrix")
    labels = sorted(y_true.unique())
    cm = confusion_matrix(y_true, y_pred, labels=labels)

    fig, ax = plt.subplots(figsize=(7, 5))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=labels, yticklabels=labels, ax=ax)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")
    ax.set_title(f"Confusion Matrix — {model_name}")
    st.pyplot(fig)

else:
    st.info("Waiting for a CSV file...")

Overwriting app.py


In [ ]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared
!streamlit run app.py &>/content/log.txt &
!sleep 5
!./cloudflared tunnel --url http://localhost:8501

2026-08-12T16:34:47Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-08-12T16:34:47Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-08-12T16:34:52Z INF +--------------------------------------------------------------------------------------------+
2026-08-12T16:34:52Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2026-08-12T16:34:52Z INF |  https://microwave-ownership-discounts-cannon.trycloud